# Implement LDA to build a Topic Model

Notes:

I am following along with my M08 LDA with SKLearn Notes and my M08 HW as I complete this section.

## Setup

### Import Libraries

In [35]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation as LDA

In [36]:
import plotly.express as px
import plotly.io as pio

sns.set_theme(style='white')
pio.renderers.default = 'vscode'

### Configuration

In [37]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [38]:
# set chapter as bag
bag = "CHAPS"

In [39]:
colors = "YlGnBu"

### Load Data

In [40]:
# load in tables
LIB = pd.read_csv('data/LIB.csv', sep='\t').set_index('book_id')
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

## Prepare Data

### Build DOCS

SKLearn expects an F1 style corpus. So I create one from the annotated CORPUS table and keep only regular nouns.

In [41]:
# only regular nouns
DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(bags[bag]).term_str\
    .apply(lambda x: ' '.join(x))\
    .to_frame('doc_str')

# DOCS

### Build Count Matrix (DTM)

Now I use SKLearn's CountVectorizer to convert the F1 corpus of chapters into a document-term vector space of word counts.

In [42]:
# instantiate CountVectorizer
count_engine = CountVectorizer(
    max_features=4000,
    max_df=.9, # drop terms appearing in more than 90% of chapters
    min_df=10, # drop terms appearing in fewer than 10 chapters (remove rare terms that would add noise without contributing to structure; hapax legomena etc)
    stop_words='english'
)

# fit the engine to the documents
# keep in mind that CountVectorizer outputs a sparse matrix
count_model = count_engine.fit_transform(DOCS['doc_str'])

# extract vocabulary array for use in PHI table
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'

# convert sparse matrix into dense Pandas dataframe
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS) # convert to DataFrame to get DTM

DTM

aback  ability  absence  absurd  \
book_id                      chap_num                                    
giants-bread                 0             0        0        0       0   
                             1             0        0        0       0   
                             2             0        0        0       0   
                             3             0        0        1       0   
                             4             0        0        0       0   
...                                      ...      ...      ...     ...   
the-seven-dials-mystery      31            1        0        0       0   
                             32            0        0        0       0   
                             33            0        0        0       0   
                             34            0        0        0       0   
the-tragedy-at-marsdon-manor 1             2        0        0       0   

                                       accent  accident  accomplice  account  \
book_id                      chap_num                                          
giants-bread                 0              0         0           0        0   
                             1              0         0           0        0   
                             2              0         0           0        0   
                             3              0         0           0        0   
                             4              0         0           0        0   
...                                       ...       ...         ...      ...   
the-seven-dials-mystery      31             0         0           0        0   
                             32             0         0           0        0   
                             33             0         1           0        0   
                             34             0         0           0        0   
the-tragedy-at-marsdon-manor 1              0         1           0        0   

                                       accounts  accuracy  ...  wound  wrist  \
book_id                      chap_num                      ...                 
giants-bread                 0                0         0  ...      0      0   
                             1                0         0  ...      0      0   
                             2                0         0  ...      0      0   
                             3                0         0  ...      0      0   
                             4                0         0  ...      0      1   
...                                         ...       ...  ...    ...    ...   
the-seven-dials-mystery      31               0         0  ...      0      0   
                             32               0         0  ...      0      0   
                             33               0         0  ...      0      0   
                             34               0         0  ...      0      0   
the-tragedy-at-marsdon-manor 1                0         0  ...      1      0   

                                       writing  yards  yawn  year  years  yes  \
book_id                      chap_num                                           
giants-bread                 0               0      0     0     0      0    0   
                             1               0      0     0     0      4    0   
                             2               0      0     0     0      0    0   
                             3               0      0     0     0      2    0   
                             4               0      0     0     0      1    0   
...                                        ...    ...   ...   ...    ...  ...   
the-seven-dials-mystery      31              0      0     0     0      0    0   
                             32              0      0     0     0      0    0   
                             33              0      0     0     0      0    0   
                             34              1      0     0     0      0    0   
the-tragedy-at-marsdon

## Fit LDA Model

In [43]:
# set parameters
n_components = 25 # number of topics
random_state = 36
n_top_terms = 5

In [44]:
# instantiate LDA model
lda_engine = LDA(
    n_components = n_components,
    random_state = random_state
)

# fit LDA model to the DTM and generate document-topic distribution
topic_dist = lda_engine.fit_transform(DTM)

# topic_dist

## Extract Results

### THETA (Document-Topic Matrix)

THETA: given this document, what is the probability distribution over topics? Each row is a document, each column is a topic, each cell is p(topic | document).

In [45]:
# create THETA (Document-Topic Matrix)
THETA = pd.DataFrame(
    topic_dist, # contains document-topic probabilities
    index = DOCS.index
)

# THETA.index.name = 'para_id' # remove this so THETA gets multindex from docs?
THETA.columns.name = 'topic_id'

# THETA

In [46]:
# THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

### PHI (Topic-Word Matrix)

PHI: given this topic, what is the probability distribution over words? Each row is a topic, each column is a term, each cell is p(word | topic).

In [47]:
# create PHI (Topic-Word Matrix)
PHI = pd.DataFrame(
    lda_engine.components_, # contains raw pseudo-counts of words to objects
    columns = TERMS
)

PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'

# PHI

In [48]:
# PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

### TOPICS (Top Terms per Topic)

Creating a TOPICS table with top 5 words per topic.

In [49]:
TOPICS = PHI.stack().groupby('topic_id').apply(
    lambda x : ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str)).to_frame("top_terms")

TOPICS

,top_terms
topic_id,
0,room sir door friend head
1,letter house head paper letters
2,door voice room moment voices
3,aback ability absence absurd accent
4,letters memoirs manuscript sir king
5,friend case crime moment head
6,things head night face people
7,door girl monsieur head face
8,house woman doctor face car


## LDA TOPIC (4)

- UVA Box URL:
- UVA Box URL of count matrix used to create:
- GitHub URL for notebook used to create:
- Delimitter:
- Libary used to compute:
- A description of any filtering, e.g. POS (Nouns and Verbs only):
- Number of components:
- Any other parameters used:
- Top 5 words and best-guess labels for topic five topics by mean document weight:
  - T00:
  - T01:
  - T02:
  - T03:
  - T04:

## Explore

### Inspect Topics

In [ ]:
TOPICS

### Topic Distributions by Book/Sleuth/Work Type

In [ ]:
#?

## LDA + PCA Visualization (4)

Apply PCA to the THETA table and plot the topics in the space opened by the first two components.

Size the points based on the mean document weight of each topic (using the THETA table).

Color the points basd on a metadata feature from the LIB table.

Provide a brief interpretation of what you see.

(INSERT IMAGE HERE)

(INSERT INTERPRETATION HERE)

## Save Outputs

In [50]:
# # save the updated VOCAB table to csv (replace the existing one created by 03-VOCAB.ipynb)
# VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)

# # save the BOW table to csv
# BOW_chaps.to_csv('data/BOW_chaps.csv', sep='\t', index=True)

# # save the DTM table to csv
# DTM.to_csv('data/DTM.csv', sep='\t', index=True)

# # save the TFIDF table to csv
# TFIDF.to_csv('data/TFIDF.csv', sep='\t', index=True)

# # save the TFIDF_L2 table to csv
# TFIDF_L2.to_csv('data/TFIDF_L2.csv', sep='\t', index=True)